# RealNVP — exact likelihoods with affine coupling layers

> Tutorial pair for [`realnvp.py`](realnvp.py).

## 1. Intuition
A normalizing flow builds a complicated distribution by pushing a simple one (a
Gaussian) through an **invertible** transformation. Because the map is invertible
and we can compute how it stretches space (its Jacobian determinant), the change-
of-variables formula gives the *exact* density of the data -- no bound (VAE) and
no adversary (GAN). RealNVP's trick is the **affine coupling layer**: transform
half the variables using the other half, so inversion and the Jacobian are both
trivial.

## 2. Concept (the slide)
- **Flow:** $z=f(x)$ invertible; base $p_Z=\mathcal N(0,I)$.
- **Density:** $\log p_X(x)=\log p_Z(f(x))+\log|\det \partial f/\partial x|$.
- **Affine coupling:** split $x=(x_a,x_b)$; keep $x_a$; map
  $x_b\mapsto x_b\odot e^{s(x_a)}+t(x_a)$.
- The Jacobian is **triangular** $\Rightarrow$ $\log|\det|=\sum s(x_a)$ -- cheap.
- Stack couplings with **alternating masks** so every coordinate gets
  transformed; sample by running the stack backward.

## 3. Math derivation — change of variables & the coupling Jacobian

**Change of variables.** If $z=f(x)$ is a diffeomorphism and $p_Z$ is the base
density, conservation of probability mass $p_X(x)|dx|=p_Z(z)|dz|$ gives
$$\boxed{\,\log p_X(x)=\log p_Z\big(f(x)\big)+\log\Big|\det\frac{\partial f}{\partial x}\Big|\,}.$$
For a composition $f=f_L\circ\cdots\circ f_1$ the log-dets simply add:
$\log|\det \partial f/\partial x|=\sum_k \log|\det \partial f_k/\partial h_{k-1}|$.

**Affine coupling layer.** Pick a binary mask $b$. Split $x=(x_a,x_b)$ with
$x_a=b\odot x$. Define
$$y_a=x_a,\qquad y_b=x_b\odot \exp\!\big(s(x_a)\big)+t(x_a),$$
where $s,t$ are arbitrary neural nets that see *only* $x_a$.

**Invertibility (no need to invert the net).** Given $y$, recover $x$ by
$$x_a=y_a,\qquad x_b=\big(y_b-t(x_a)\big)\odot\exp\!\big(-s(x_a)\big).$$
Because $s,t$ are evaluated at $x_a=y_a$ in both directions, we never invert the
networks themselves -- they can be arbitrarily complex.

**The Jacobian is triangular.** Order coordinates $(x_a,x_b)$. Then
$$\frac{\partial(y_a,y_b)}{\partial(x_a,x_b)}=
\begin{pmatrix} I & 0\\[2pt] \dfrac{\partial y_b}{\partial x_a} & \mathrm{diag}(e^{s(x_a)})\end{pmatrix}.$$
The matrix is block lower-triangular, so its determinant is the product of the
diagonal blocks; the off-diagonal $\partial y_b/\partial x_a$ is irrelevant:
$$\boxed{\,\log\Big|\det\frac{\partial y}{\partial x}\Big|=\sum_j s(x_a)_j\,}.$$
This is the whole reason RealNVP scales: the log-det that would normally cost
$O(d^3)$ is just a sum.

**Alternating masks.** A single coupling leaves $x_a$ untouched, so we stack
several and flip the mask each time; after a few layers every coordinate has been
both "kept" and "transformed", letting the flow model arbitrary couplings.

**Training.** Maximize the exact log-likelihood (equivalently minimize NLL):
$$\mathcal L=-\frac1N\sum_i\Big[\log\mathcal N\big(f(x_i);0,I\big)+\sum_k\log|\det J_k(x_i)|\Big].$$

## 4. Model — affine coupling layer + the RealNVP stack

In [ ]:
# ===== actual implementation from realnvp.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _coupling_logdet_numpy(s):
    """log|det J| of one affine coupling = sum of the log-scales s."""
    return np.sum(s, axis=-1)

import torch

import torch.nn as nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class AffineCoupling(nn.Module):
    r"""
    One affine coupling layer with a fixed binary mask b in {0,1}^d.

    Forward (data -> latent):
        x_a = b * x                      (kept unchanged)
        s, t = NN(x_a)                   (computed from the kept part only)
        y    = x_a + (1-b) * ( x * exp(s) + t )
        log|det| = sum over the transformed dims of s
    Inverse (latent -> data) just solves the affine map; the NN sees x_a = b*y = b*x
    unchanged, so it is exactly invertible without inverting the NN.
    """

    def __init__(self, dim: int, mask: torch.Tensor, hidden: int = 64):
        super().__init__()
        self.register_buffer("mask", mask)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2 * dim))           # outputs (s, t) stacked
        self.dim = dim
        # scale the raw log-scale by a learned factor + tanh -> bounded, stable
        self.scale = nn.Parameter(torch.zeros(dim))

    def _st(self, x_a: torch.Tensor):
        h = self.net(x_a)
        s, t = h.chunk(2, dim=1)
        s = torch.tanh(s) * self.scale            # bounded log-scale
        return s, t

    def forward(self, x: torch.Tensor):
        """data -> latent; returns (y, log|det J|)."""
        x_a = x * self.mask
        s, t = self._st(x_a)
        s = s * (1 - self.mask); t = t * (1 - self.mask)   # only transform the other half
        y = x_a + (1 - self.mask) * (x * torch.exp(s) + t)
        log_det = s.sum(dim=1)
        return y, log_det

    def inverse(self, y: torch.Tensor):
        """latent -> data (exact)."""
        y_a = y * self.mask
        s, t = self._st(y_a)
        s = s * (1 - self.mask); t = t * (1 - self.mask)
        x = y_a + (1 - self.mask) * ((y - t) * torch.exp(-s))
        return x

class RealNVP(nn.Module):
    """A small stack of affine couplings with alternating masks, N(0,I) base."""

    def __init__(self, dim: int = 2, n_couplings: int = 6, hidden: int = 64):
        super().__init__()
        masks = []
        for i in range(n_couplings):
            m = torch.zeros(dim)
            m[i % 2::2] = 1.0                      # alternate which half is kept
            masks.append(m)
        self.layers = nn.ModuleList(
            AffineCoupling(dim, masks[i], hidden) for i in range(n_couplings))
        self.dim = dim

    def forward(self, x: torch.Tensor):
        """data -> latent z, accumulating the total log|det|."""
        log_det = torch.zeros(len(x), device=x.device)
        z = x
        for layer in self.layers:
            z, ld = layer(z)
            log_det = log_det + ld
        return z, log_det

    def inverse(self, z: torch.Tensor):
        """latent z -> data x (run the stack backward)."""
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """Exact log p(x) = log N(z;0,I) + log|det dz/dx|  (change of variables)."""
        z, log_det = self(x)
        base = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(dim=1)   # standard normal
        return base + log_det

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 5e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                nll = -self.log_prob(X[perm[s:s + batch]]).mean()
                opt.zero_grad(); nll.backward(); opt.step()
                tot += nll.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int):
        dev = next(self.parameters()).device
        z = torch.randn(n, self.dim, device=dev)
        return self.inverse(z).cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import make_moons
    X, _ = make_moons(2000, noise=0.05, random_state=SEED)
    X = ((X - X.mean(0)) / X.std(0)).astype(np.float32)

    m = RealNVP(dim=2, n_couplings=6).fit(X, epochs=400)
    ll = m.log_prob(torch.tensor(X)).mean().item()
    print(f"RealNVP final NLL = {m.history[-1]:.3f}  (mean log-lik = {ll:.3f})")

    s = m.sample(2000)
    print(f"  data    mean={X.mean(0).round(2)}  std={X.std(0).round(2)}")
    print(f"  samples mean={s.mean(0).round(2)}  std={s.std(0).round(2)}")

## 5. Training / sampling — exact log_prob, NLL training, inverse sampler

In [ ]:
# ===== actual implementation from realnvp.py =====

## 6. Train & sample on 2-D two-moons (exact likelihood)

In [ ]:
demo()

## 7. Visualization — learned density and samples

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt, torch
from sklearn.datasets import make_moons
import realnvp as M

X, _ = make_moons(2000, noise=0.05, random_state=0)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
m = M.RealNVP(dim=2, n_couplings=6).fit(X, epochs=400)

# evaluate the exact learned density on a grid
gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 120), np.linspace(-2.5, 2.5, 120))
grid = np.c_[gx.ravel(), gy.ravel()].astype("float32")
with torch.no_grad():
    logp = m.log_prob(torch.tensor(grid)).numpy().reshape(gx.shape)
s = m.sample(2000)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].contourf(gx, gy, np.exp(logp), levels=30, cmap="viridis")
ax[0].scatter(X[:, 0], X[:, 1], s=3, alpha=.2, color="white")
ax[0].set_title("Exact learned density p(x)"); ax[0].set_aspect("equal")
ax[1].scatter(X[:, 0], X[:, 1], s=4, alpha=.3, label="data")
ax[1].scatter(s[:, 0], s[:, 1], s=4, alpha=.4, color="r", label="flow samples")
ax[1].legend(); ax[1].set_title("Samples (z~N(0,I) run backward)"); ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Flows give an **exact** tractable likelihood -- the change-of-variables formula
  is the entire engine.
- The coupling layer's **triangular Jacobian** makes the log-det a cheap sum;
  this is what lets flows scale beyond toy dimensions.
- Use **alternating masks** so every coordinate is eventually transformed.
- Pitfalls: unbounded log-scales $s$ blow up -- bound them (here with
  $\tanh$); the dimension must be preserved (flows are bijections, so no
  bottleneck/compression).
- **Glow** (next file) generalizes the fixed mask into a learned invertible
  $1\times1$ convolution and adds actnorm.